# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [13]:
import warnings
warnings.filterwarnings('ignore')

In [15]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [16]:
import pandas as pd
df = pd.read_csv('/Users/domiendarmont/Desktop/Ironhack/lab_week7_lang/lab-chains-in-langchain/data/Data.csv')

In [17]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

In [6]:
!pip install langchain langchain-core langchain-openai langchain-community langchain-classic

  Using cached langchain-1.2.18-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langgraph-1.1.10-py3-none-any.whl.metadata (8.0 kB)
  Using cached langgraph_checkpoint-4.0.3-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_prebuilt-1.0.13-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.3.14-py3-none-any.whl.metadata (1.7 kB)
  Using cached ormsgpack-1.12.2-cp314-cp314-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (3.2 kB)
  Using cached aiohttp-3.13.5-cp314-cp314-macosx_11_0_arm64.whl.metadata (8.1 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.14.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Us

In [7]:
!pip show langchain langchain-core langchain-openai langchain-community

Name: langchain
Version: 1.2.18
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /Users/domiendarmont/Desktop/Ironhack/lab_week7_lang/lab-chains-in-langchain/langchain/lib/python3.14/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-core
Version: 1.3.3
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /Users/domiendarmont/Desktop/Ironhack/lab_week7_lang/lab-chains-in-langchain/langchain/lib/python3.14/site-packages
Requires: jsonpatch, langchain-protocol, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-classic, langchain-community, langchain-openai, langchain-text-splitters, langgraph, langgraph-checkpoint, langgraph-prebuilt
---
Name: langchain-openai
Version: 1.2.1
Summary: An int

In [8]:
from langchain_openai import ChatOpenAI        # ✅ 1.0.3
from langchain_core.prompts import ChatPromptTemplate  # ✅ From langchain-core 1.0.5
from langchain_classic.chains import LLMChain  # ✅ Legacy (install if missing)

In [19]:
#Replace None by your own value and justify
llm = ChatOpenAI(temperature=0.7)


In [67]:
prompt = ChatPromptTemplate.from_template("write a 10 word discription for a 37year old about {product}"
 
)

In [68]:

chain = LLMChain(llm=llm, prompt=prompt)

In [66]:
product = "walkman"
chain.run(product)

'Retro portable music player that defined the 80s generation. Enjoy!'

## SimpleSequentialChain

In [29]:
from langchain_classic.chains import SimpleSequentialChain  

In [69]:
llm = ChatOpenAI(temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template("write a 10 word discription for a 37year old about {walkman}")

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)



In [70]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template("summarize this all to 5 words: {text}")

# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [71]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [72]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
Nostalgic portable cassette player for 80s music lovers on-the-go.
Portable cassette player for nostalgia.

> Finished chain.


'Portable cassette player for nostalgia.'

**Repeat the above twice for different products**

## SequentialChain

In [34]:
from langchain_classic.chains import SequentialChain  

In [73]:
llm = ChatOpenAI(temperature=0.9)


first_prompt = ChatPromptTemplate.from_template("Say something about this product in 15 words in Dutch: {topic}")
topic = "safety razor for men"
chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="description_in_Dutch"
                    )

In [75]:
second_prompt = ChatPromptTemplate.from_template("summarize this all to 5 words: {description_in_Dutch}")

chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summary"
                    )

In [76]:
# prompt template 3: translate to english or other language
third_prompt = ChatPromptTemplate.from_template("translate this to English: {summary}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="summary_in_English"
                      )



In [77]:
# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
        "Based on the following information, provide a final summary in 15 words: {summary_in_English}"
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="final_summary"
                     )



In [78]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["topic"],
    output_variables=["description_in_Dutch", "summary", "summary_in_English", "final_summary"],
    verbose=True
)



In [ ]:
review = df.Review[5]
overall_chain(review)



> Entering new SequentialChain chain...

> Finished chain.


{'topic': 'I loved the waterproof sac, although the opening was made of a hard plastic. I don’t know if that would break easily. But I couldn’t turn my phone on, once it was in the pouch.',
 'description_in_Dutch': 'De waterdichte zak was geweldig, maar de harde plastic opening maakte me bezorgd over breuk.',
 'summary': 'Goede waterdichte tas, harde opening.',
 'summary_in_English': 'Good waterproof bag, hard opening.',
 'final_summary': 'Waterproof bag with tough opening, reliable for protecting valuables in any weather condition.'}

In [84]:
review = df.Product[0]
overall_chain(review)



> Entering new SequentialChain chain...

> Finished chain.


{'topic': 'Queen Size Sheet Set',
 'description_in_Dutch': 'Een luxe queensize laken set voor een heerlijke nachtrust in stijl en comfort.',
 'summary': 'Luxe queensize laken set comfortabel.',
 'summary_in_English': 'Luxury queen-size sheet set comfortable.',
 'final_summary': "Luxury queen-size sheet set delivers ultimate comfort for a restful night's sleep."}

In [85]:
review = df.Review[0]
overall_chain(review)



> Entering new SequentialChain chain...

> Finished chain.


{'topic': 'I ordered a king size set. My only criticism would be that I wish seller would offer the king size set with 4 pillowcases. I separately ordered a two pack of pillowcases so I could have a total of four. When I saw the two packages, it looked like the color did not exactly match. Customer service was excellent about sending me two more pillowcases so I would have four that matched. Excellent! For the cost of these sheets, I am satisfied with the characteristics and coolness of the sheets.',
 'description_in_Dutch': 'Ik heb een kingsize set besteld. Het enige minpunt is dat ik wou dat de verkoper ook een kingsize set met 4 kussenslopen zou aanbieden. Ik heb apart een tweepak kussenslopen besteld zodat ik er toch vier zou hebben. Toen ik de twee pakketten zag, leek het alsof de kleur niet helemaal overeenkwam. De klantenservice was uitstekend in het sturen van mij nog twee kussenslopen zodat ik vier gelijke had. Uitstekend! Voor de prijs van deze lakens ben ik tevreden met de e

**Repeat the above twice for different products or reviews**

## Router Chain

In [86]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [87]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [88]:
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain_core.prompts import PromptTemplate  # Updated path for prompts (core package)

In [89]:
llm = ChatOpenAI(temperature=0)

In [90]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [91]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [92]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [93]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [94]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [95]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation refers to the electromagnetic radiation emitted by a perfect black body, which is an idealized physical body that absorbs all incident electromagnetic radiation and emits radiation at all frequencies. The radiation emitted by a black body depends only on its temperature and follows a specific distribution known as Planck's law. This type of radiation is important in understanding concepts such as thermal radiation and the behavior of objects at different temperatures."

In [96]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'The answer to 2 + 2 is 4.'

In [97]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


"Every cell in our body contains DNA because DNA is the genetic material that carries the instructions for the development, functioning, and reproduction of all living organisms. DNA contains the information needed to build and maintain an organism, including the proteins that make up our cells and tissues. \n\nHaving DNA in every cell ensures that each cell has the necessary genetic information to carry out its specific functions and to replicate itself accurately during cell division. This ensures that the genetic information is passed on to the next generation of cells, maintaining the integrity and continuity of the organism's genetic code.\n\nAdditionally, DNA serves as a storage system for genetic information that can be accessed and utilized by cells as needed. This allows for the regulation of gene expression, the repair of damaged DNA, and the adaptation to changing environmental conditions.\n\nIn summary, every cell in our body contains DNA because it is essential for the pro

**Repeat the above at least once for different inputs and chains executions - Be creative!**